In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate

def c_amodN(a, N, power, n_qubits):
    dim = 2**n_qubits
    U = np.zeros((dim, dim))
    
    for y in range(dim):
        if y < N:
            new_y = (y * pow(a, power, N)) % N
        else:
            new_y = y  # leave invalid states unchanged
        U[new_y, y] = 1

    gate = UnitaryGate(U, label=f"{a}^{power} mod {N}")
    return gate.control()

In [2]:
import numpy as np
from qiskit import QuantumCircuit

def qft_dagger(n):
    qc = QuantumCircuit(n)

    # Reverse qubit order
    for qubit in range(n // 2):
        qc.swap(qubit, n - qubit - 1)

    # Apply inverse QFT
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / (2 ** (j - m)), m, j)
        qc.h(j)

    qc.name = "QFT†"
    return qc

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import UnitaryGate
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from fractions import Fraction

# -----------------------------
# PARAMETERS
# -----------------------------
N = 5
a = 2
N_COUNT = 5
N_WORK = 3
# -----------------------------
# BUILD CIRCUIT
# -----------------------------
qc = QuantumCircuit(N_COUNT + N_WORK, N_COUNT)

# superposition
for q in range(N_COUNT):
    qc.h(q)

# |1>
qc.x(N_COUNT)

# controlled multiplications
for q in range(N_COUNT):
    qc.append(
        c_amodN(a, N, 2**q, N_WORK),
        [q] + [i+N_COUNT for i in range(N_WORK)]
    )

# inverse QFT
qc.append(qft_dagger(N_COUNT), range(N_COUNT))

# measurement
qc.measure(range(N_COUNT), range(N_COUNT))

# -----------------------------
# RUN
# -----------------------------
simulator = AerSimulator()
tcircuit = transpile(qc, simulator)

job = simulator.run(tcircuit, shots=1024)
result = job.result()
counts = result.get_counts()

# -----------------------------
# RESULTS
# -----------------------------
print("Counts:", counts)

plot_histogram(counts)
plt.show()

print("\nPhase estimation:")
for output in counts:
    decimal = int(output, 2)
    phase = decimal / (2 ** N_COUNT)
    frac = Fraction(phase).limit_denominator(N)

    print(f"{output} -> phase={phase:.4f} -> {frac} -> r={frac.denominator}")

Counts: {'00000': 242, '01000': 250, '10000': 272, '11000': 260}

Phase estimation:
00000 -> phase=0.0000 -> 0 -> r=1
01000 -> phase=0.2500 -> 1/4 -> r=4
10000 -> phase=0.5000 -> 1/2 -> r=2
11000 -> phase=0.7500 -> 3/4 -> r=4


In [5]:
from math import gcd

# -----------------------------
# FUNCTION TO FIND FACTORS
# -----------------------------
def shor_factors(N, a, r):
    """
    Given N, coprime a, and period r,
    attempt to find non-trivial factors of N.
    Returns a list of factors (or empty list if none found).
    """
    if r % 2 != 0:
        # period must be even
        return []

    possible_factors = []
    ar2 = pow(a, r // 2, N)
    
    # compute gcd(a^(r/2) ± 1, N)
    f1 = gcd(ar2 - 1, N)
    f2 = gcd(ar2 + 1, N)

    # check for non-trivial
    for f in [f1, f2]:
        if f != 1 and f != N:
            possible_factors.append(f)

    return possible_factors

# -----------------------------
# EXAMPLE USAGE
# -----------------------------
# assume r=4 from phase estimation, a=2, N=5
r = 4
factors = shor_factors(N, a, r)
if factors:
    print(f"Non-trivial factors of {N} found: {factors}")
else:
    print(f"No non-trivial factors found for a={a}, r={r}. Try another a.")

# -----------------------------
# TRY ANOTHER COPRIME a=3
# -----------------------------
a = 3
# Manually compute period r for a=3 mod 5 (or use your quantum routine)
r = 4  # known period for 3^x mod 5
factors = shor_factors(N, a, r)
if factors:
    print(f"Non-trivial factors of {N} found with a={a}: {factors}")
else:
    print(f"No non-trivial factors found for a={a}, r={r}.")

No non-trivial factors found for a=2, r=4. Try another a.
No non-trivial factors found for a=3, r=4.
